# Task 5 density + autoencoder submission and CSV comparison

This freezes Tasks 1–4 from the locally scored Tail v20 file, replaces only Task 5 with the saved density + autoencoder ranking, and uses the friend's 2,046 alert count as an equal-budget comparison. The friend's CSV is prediction evidence only; it is not used in fitting or ranking. Kaggle's 0.81771 is an overall score of the teammate's five-task file. The user reports that earlier Task 1–4 experiments contributed little to its jump, making Task 5 the likely main driver, though this file alone cannot quantify the Task 5 delta.

In [1]:
from pathlib import Path
import sys, json, hashlib, joblib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/pipeline/save_model.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
import numpy as np
import pandas as pd
from src.pipeline.random_seed import set_seed
SEED = set_seed()
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
BASE = SRC / "submission/kaggle_t12345_t5_tail_engineered_v20.csv"
FRIEND = Path.home() / "Downloads" / "kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_density_auto_v1.csv"
base = pd.read_csv(BASE, dtype=str)
friend = pd.read_csv(FRIEND, dtype=str)
sample = pd.read_csv(SRC / "submission/sample_submission.csv", usecols=["id"], dtype=str)
assert base.columns.tolist() == friend.columns.tolist() == ["id", "task1", "task2", "task3", "task4", "task5"]
assert len(base) == len(friend) == len(sample) == 25889
assert base.id.equals(friend.id) and base.id.equals(sample.id) and base.id.is_unique
assert not base.isna().any().any() and not friend.isna().any().any()
base_card = json.loads(BASE.with_suffix(".json").read_text(encoding="utf-8"))
assert base_card["output_sha256"] == sha(BASE)
print("Base", BASE.name, "SHA-256", sha(BASE))

Base kaggle_t12345_t5_tail_engineered_v20.csv SHA-256 2dd79b546af197688d3ff35211cc744c2e86a28360ea3a01343f743364c1505a


In [2]:
MODEL_NAME = "t5_density_auto_original_v1"
model_card_path = SRC / "task5/save" / f"{MODEL_NAME}.json"
model_card = json.loads(model_card_path.read_text(encoding="utf-8"))
assert model_card["data_name"] == "eda_original_v2"
model = joblib.load(SRC / model_card["model_path"])
data_card_path = SRC / "task5/data/eda" / f"{model_card['data_name']}.json"
data_card = json.loads(data_card_path.read_text(encoding="utf-8"))
frame = pd.read_parquet(SRC / data_card["splits"]["test"]["path"])
assert len(frame) == len(base) and frame.player_id.is_unique
X = frame.drop(columns="player_id")
scores = model.anomaly_score(X)
density_score, reconstruction_score = model.component_scores(X)
assert len(scores) == len(base) and np.isfinite(scores).all()
k = int(friend.task5.astype(int).sum())
assert k == 2046
chosen = np.lexsort((np.arange(len(scores)), -scores))[:k]
pred = np.zeros(len(scores), dtype=np.int8); pred[chosen] = 1
assert int(pred.sum()) == k

In [3]:
NAME = "kaggle_t12345_t5_density_auto_v24"
out_path = SRC / "submission" / f"{NAME}.csv"
meta_path = out_path.with_suffix(".json")
output = base.copy(); output["task5"] = pred.astype(str)
assert output[["id","task1","task2","task3","task4"]].equals(base[["id","task1","task2","task3","task4"]])
assert output.id.equals(sample.id) and not output.isna().any().any()
base_pred = base.task5.astype(int).to_numpy()
friend_pred = friend.task5.astype(int).to_numpy()
report = pd.DataFrame({"id":base.id, "friend_task5":friend_pred, "ours_task5":pred,
                       "tail_v20_task5":base_pred, "density_rarity":density_score,
                       "autoencoder_error":reconstruction_score, "blend_rank":scores})
report_path = SRC / "task5/models/density_auto_comparison_v1.csv"
detail = {"name":NAME,"status":"prepared locally; not uploaded", "base_submission":BASE.name,
          "base_sha256":sha(BASE), "friend_comparison_file":FRIEND.name,
          "friend_sha256":sha(FRIEND), "comparison_budget":k,"alerts":int(pred.sum()),
          "changed_task":"task5", "task5_orientation":"1 = anomaly",
          "score_formula":"0.5 * fitted density rarity percentile + 0.5 * fitted autoencoder error percentile; select 2046 largest, row index breaks ties",
          "model_card":model_card_path.relative_to(SRC).as_posix(),
          "dataset_card":data_card_path.relative_to(SRC).as_posix(),
          "rows":len(output), "seed":SEED}
if out_path.exists() or meta_path.exists() or report_path.exists():
    if not all(p.exists() for p in (out_path, meta_path, report_path)):
        raise RuntimeError("Partial output exists; inspect before rerunning")
    assert pd.read_csv(out_path,dtype=str).equals(output)
    assert json.loads(meta_path.read_text(encoding="utf-8"))["output_sha256"] == sha(out_path)
else:
    output.to_csv(out_path,index=False)
    detail["output_sha256"] = sha(out_path)
    meta_path.write_text(json.dumps(detail,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    report.to_csv(report_path,index=False)
print("Saved",out_path,"SHA-256",sha(out_path))
print("Comparison",report_path)
print("Friend vs ours Task 5:\n",pd.crosstab(friend_pred,pred,rownames=["friend"],colnames=["ours"]))
print("Tail v20 vs ours Task 5:\n",pd.crosstab(base_pred,pred,rownames=["tail"],colnames=["ours"]))
print("Friend vs base Tasks 1–4 equal rows (numeric values):", {c:int((pd.to_numeric(friend[c])==pd.to_numeric(base[c])).sum()) for c in ["task1","task2","task3","task4"]})

Saved C:\Users\BTCOM\Desktop\ml-final\src\submission\kaggle_t12345_t5_density_auto_v24.csv SHA-256 241c1c0c14ddbbe412866a63f671c6199e373396bda4b545ac4fb52517e95ea0
Comparison C:\Users\BTCOM\Desktop\ml-final\src\task5\models\density_auto_comparison_v1.csv
Friend vs ours Task 5:
 ours        0     1
friend             
0       22375  1468
1        1468   578
Tail v20 vs ours Task 5:
 ours      0     1
tail             
0     18432   926
1      5411  1120
Friend vs base Tasks 1–4 equal rows (numeric values): {'task1': 24620, 'task2': 25889, 'task3': 1719, 'task4': 21172}
